In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from geopy.geocoders import Nominatim
from geopy.exc import GeocoderTimedOut
import time

# Carica il dataset
df = pd.read_csv('../etl/dataset-school-clean-grouped.csv', sep=';')
print("Dataset caricato:")
print(df.head())
print(f"\nShape: {df.shape}")
print(f"Colonne: {df.columns.tolist()}")

In [ ]:
# Coordinate centroidi dei paesi africani (no API calls)
country_centroids = {
    "Angola":           (-11.2027, 17.8739),
    "Algeria":          (28.0339, 1.6596),
    "Benin":            (9.3077,   2.3158),
    "Burkina Faso":     (12.3641,  -1.5275),
    "Burundi":          (-3.3731,  29.9189),
    "Cameroon":         (3.8480,   11.5021),
    "Cape Verde":       (16.5388,  -23.0418),
    "C. A. R.":         (6.6111,   20.9394),
    "Central African Republic": (6.6111, 20.9394),
    "Chad":             (15.4542,  18.7322),
    "Comoros":          (-11.6455, 43.3333),
    "Congo":            (-0.2280,  15.8277),
    "Cote d'Ivoire":    (7.5400,   -5.5471),
    "Côte d'Ivoire":    (7.5400,   -5.5471),
    "D. R. Congo":      (-4.0383,  21.7587),
    "Dem. Rep. Congo":  (-4.0383,  21.7587),
    "Djibouti":         (11.8251,  42.5903),
    "Egypt":            (26.8206,  30.8025),
    "Equat. Guinea":    (1.6508,   10.2679),
    "Eritrea":          (15.1794,  39.7823),
    "Eswatini":         (-26.5225, 31.4659),
    "Ethiopia":         (9.1450,   40.4897),
    "Gabon":            (-0.8037,  11.6094),
    "Gambia":           (13.4432,  -15.3101),
    "Ghana":            (7.9465,   -1.0232),
    "Guinea":           (9.9456,   -11.2788),
    "Guinea-Bissau":    (11.8037,  -15.1804),
    "Kenya":            (-0.0236,  37.9062),
    "Lesotho":          (-29.6100, 28.2336),
    "Liberia":          (6.4281,   -9.4295),
    "Libya":            (26.3351,  17.2283),
    "Madagascar":       (-18.7669, 46.8691),
    "Malawi":           (-13.2543, 34.3015),
    "Mali":             (17.5707,  -3.9962),
    "Mauritania":       (21.0079,  -10.9408),
    "Mauritius":        (-20.3484, 57.5522),
    "Morocco":          (31.7917,  -7.0926),
    "Mozambique":       (-18.6657, 35.5296),
    "Namibia":          (-22.9576, 18.4904),
    "Niger":            (17.6078,  8.0817),
    "Nigeria":          (9.0820,   8.6753),
    "Rwanda":           (-1.9403,  29.8739),
    "S. Tome/Principe": (0.1864,   6.6131),
    "Sao Tome & Principe": (0.1864, 6.6131),
    "Senegal":          (14.4974,  -14.4524),
    "Sierra Leone":     (8.4606,   -11.7799),
    "Somalia":          (5.1521,   46.1996),
    "South Africa":     (-30.5595, 22.9375),
    "South Sudan":      (6.8770,   31.3070),
    "Sudan":            (12.8628,  30.2176),
    "Swaziland":        (-26.5225, 31.4659),
    "Tanzania":         (-6.3690,  34.8888),
    "U. R. Tanzania":   (-6.3690,  34.8888),
    "Togo":             (8.6195,   0.8248),
    "Tunisia":          (33.8869,  9.5375),
    "Uganda":           (1.3733,   32.2903),
    "Zambia":           (-13.1339, 27.8493),
    "Zimbabwe":         (-19.0154, 29.1549),
}

# Assegna coordinate: centroide del paese + piccolo offset casuale per regione
rng = np.random.default_rng(42)  # seed fisso per riproducibilità

def assign_coords(row):
    country = row['country']
    if country in country_centroids:
        lat, lon = country_centroids[country]
        # Offset casuale entro ±2 gradi per separare le regioni
        lat += rng.uniform(-2.0, 2.0)
        lon += rng.uniform(-2.0, 2.0)
        return pd.Series([lat, lon])
    return pd.Series([None, None])

df[['latitude', 'longitude']] = df.apply(assign_coords, axis=1)

found = df[['latitude', 'longitude']].notna().all(axis=1).sum()
missing_countries = df[df['latitude'].isna()]['country'].unique()
print(f"Righe con coordinate: {found}/{len(df)}")
if len(missing_countries) > 0:
    print(f"Paesi non trovati nel mapping: {missing_countries}")
print(df.head(10))

In [ ]:
# Filtra righe con coordinate o metriche mancanti e rimuovi outlier
df_clean = df.dropna(subset=['latitude', 'longitude', 'compl_primary_avg']).copy()

# Valori validi: compl_primary_avg deve essere tra 0 e 1
outliers = df_clean[df_clean['compl_primary_avg'] > 1.0]
print(f"Outlier rimossi (compl_primary_avg > 1): {len(outliers)} righe")
if len(outliers) > 0:
    print(outliers[['country', 'region', 'compl_primary_avg']].head(5))

df_clean = df_clean[df_clean['compl_primary_avg'] <= 1.0]

print(f"\nDati validi per il plot: {len(df_clean)}/{len(df)}")
print(f"\nStatistiche di compl_primary_avg:\n{df_clean['compl_primary_avg'].describe()}")

In [ ]:
# Crea il hexbin plot
fig = go.Figure()

# Aggiungi il hexbin layer
fig.add_trace(go.Scatterdensity(
    x=df_clean['longitude'],
    y=df_clean['latitude'],
    z=df_clean['compl_primary_avg'],
    mode='markers',
    marker=dict(
        size=8,
        color=df_clean['compl_primary_avg'],
        colorscale='Viridis',
        showscale=True,
        colorbar=dict(
            title="Completion Rate<br>(Primary)",
            thickness=15,
            len=0.7,
        ),
        opacity=0.7,
        line=dict(width=0)
    ),
    text=[f"{row['country']}<br>{row['region']}<br>Completion: {row['compl_primary_avg']:.2%}" 
          for _, row in df_clean.iterrows()],
    hovertemplate='%{text}<extra></extra>',
    name=''
))

# Layout
fig.update_layout(
    title="Hexbin Plot: Primary School Completion Rate by Region",
    xaxis_title="Longitude",
    yaxis_title="Latitude",
    hovermode='closest',
    width=1200,
    height=700,
    template='plotly_white',
    font=dict(size=12)
)

fig.show()

import matplotlib.pyplot as plt
import geopandas as gpd

# Carica il world shapefile e filtra l'Africa
world = gpd.read_file(gpd.datasets.get_path('naturalearth_lowres'))
africa = world[world['continent'] == 'Africa']

fig_mpl, ax = plt.subplots(figsize=(14, 10))

# Sfondo: contorno dell'Africa
africa.plot(
    ax=ax,
    color='#f5f0e8',   # beige chiaro per il territorio
    edgecolor='#aaaaaa',
    linewidth=0.6,
    zorder=1
)

# Hexbin plot sovrapposto
hb = ax.hexbin(
    df_clean['longitude'],
    df_clean['latitude'],
    C=df_clean['compl_primary_avg'],
    gridsize=20,
    cmap='viridis',
    mincnt=1,
    edgecolors='white',
    linewidths=0.3,
    reduce_C_function=np.mean,
    alpha=0.85,
    zorder=2
)

# Limita la vista all'Africa
ax.set_xlim(-25, 55)
ax.set_ylim(-38, 40)

ax.set_xlabel('Longitude', fontsize=12)
ax.set_ylabel('Latitude', fontsize=12)
ax.set_title('Hexbin Plot: Mean Primary School Completion Rate by Geographic Region', fontsize=14, fontweight='bold')

cbar = plt.colorbar(hb, ax=ax)
cbar.set_label('Completion Rate (Primary)', fontsize=11)

plt.tight_layout()
plt.show()

print(f"\nRiepilogo visualizzazione:")
print(f"- Totale aree geografiche: {len(df_clean)}")
print(f"- Completion rate medio: {df_clean['compl_primary_avg'].mean():.2%}")
print(f"- Range: {df_clean['compl_primary_avg'].min():.2%} - {df_clean['compl_primary_avg'].max():.2%}")